In [ ]:
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [ ]:
ft_model_name = 'Qwen3-8B'
# ft_model_name = 'Qwen3-30B-A3B'
# ft_model_name = 'gpt-oss-20b'

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, TrainingArguments
from datasets import Dataset

import pandas as pd
import torch

In [ ]:
data_root = './datasets/huanhuan_data/huanhuan.json'
df = pd.read_json(data_root)
df

In [ ]:
datas = Dataset.from_pandas(df)
datas = datas.train_test_split(test_size=0.2)
datas

In [ ]:
if ft_model_name == 'Qwen3-8B':
    qwen3_model_name = 'Qwen/Qwen3-8B'

print(f'qwen3_model_name is {qwen3_model_name}')

tokenizer = AutoTokenizer.from_pretrained(qwen3_model_name, use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(qwen3_model_name, device_map='auto', torch_dtype=torch.bfloat16)

In [ ]:
model.enable_input_require_grads()

In [ ]:
messages = [
        {'role':'system','content':'===system_message_test==='},
        {'role':'user','content':'===user_message_test==='},
        {'role':'assistant','content':'===assistant_message_test==='}
    ]

# 应用该模板，生成格式化的文本
text = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, # 表示先不进行分词，然后只返回文本
    add_generation_prompt=True, # 表示在生成文本时，添加一个特殊的标记，用于表示生成的文本开始
    enable_thinking=False # 是否启用思考模式
)
print(f'测试：生成的格式化文本为：{text}')

In [ ]:
def process_func(example):
    '''
    对数据进行预处理
    :param example: 数据集的一条数据
    :return: 预处理后的数据集
    '''

    # 1.设置数据的最大序列长度，上下文窗口大小
    MAX_LENGTH = 1024

    # 2.初始化返回值列表
    input_ids, attention_mask, labels = [], [], []

    # 3.构建instruction，适配刚才构建好的文本模板格式
    instruction = tokenizer(
        f"<s><|im_start|>system\n现在你要扮演皇帝身边的女人--甄嬛<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction'] + example['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens=False,
    )

    # 4.构建response部分
    response = tokenizer(
        f"{example['output']}",
        add_special_tokens=False,
    )

    # 5.将instruction部分和response部分的input_ids拼接，然后末尾添加pad_token作为结束符
    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.pad_token_id]

    # 6.构建attention_mask，1表示参与计算，0表示不参与计算
    attention_mask = instruction['attention_mask'] + response['attention_mask'] + [1]

    # 7.构建标记labels，-100表示不计算损失
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.pad_token_id]

    # 8.如果序列长度超过上下文窗口大小，则截断
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    # 9.返回处理后的数据集
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [ ]:
dataset_ds = datas.map(process_func, remove_columns=datas['train'].column_names)
print(f'测试：预处理后的训练数据集为：{len(dataset_ds['train'])}, 预处理后的测试数据集为：{len(dataset_ds['test'])}')
print(f'测试：第一条样本的input_ids为：{dataset_ds['train'][0]["input_ids"]}')
print(f'测试：第一条样本经由tokenizer的decode解码后的文本内容为：{tokenizer.decode(dataset_ds['train'][0]["input_ids"])}')

In [ ]:
config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, # 任务类型，因为当前Qwen3-8B本质是一个LLMs，还是基于Decoder-Only架构设计的，所以这里表示任务类型是一个因果语言模型的训练
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], # 目标模块，这里表示Qwen3-8B模型中的注意力层和MLP层
        inference_mode=False, # 推理模式，False表示训练模式，True表示推理模式
        r=8, # lora低参微调的秩，这里表示低参微调的秩为8
        lora_alpha=32, # lora低参微调的缩放系数，这里表示缩放系数为32
        lora_dropout=0.1, # lora低参微调的dropout概率，这里表示dropout概率为0.1
    )

In [ ]:
model = get_peft_model(model, config)

model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
        output_dir=f'./outputs/{ft_model_name}/', # 输出目录
        per_device_train_batch_size=8, # 每个设备的训练批量大小
        gradient_accumulation_steps=4, # 梯度累加步数
        logging_steps=10, # 日志打印步数
        num_train_epochs=3, # 训练轮数
        save_steps=100, # 模型保存步数
        learning_rate=1e-4, # 学习率
        save_on_each_node=True, # 每个节点都保存一次模型
        gradient_checkpointing=True # 梯度检查点
    )

In [ ]:
trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_ds['train'],
        eval_dataset=dataset_ds['test'],
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
    )

In [ ]:
trainer.train()

In [1]:
mode_path = '../source/lesson_models/Qwen3-8B'
lora_path = './outputs/Qwen3-8B/checkpoint-351' # 这里改称你的 lora 输出对应 checkpoint 地址

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(mode_path, use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(mode_path, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)

In [ ]:
model = PeftModel.from_pretrained(model, model_id=lora_path)
print('\U0001F60D哈喽，我是基于Qwen3-8B微调的甄嬛体问答模型，我可以用甄嬛体来回答你的问题哦~')

In [ ]:
print('\U0001F60D哈喽，我是基于Qwen3-8B微调的甄嬛体问答模型，我可以用甄嬛体来回答你的问题哦~')
while True:
    prompt = input('\U0001F600我是嬛嬛，你请说：')
    if prompt == 'exit':
        print('\U0001F62D好的拜拜，欢迎再次使用~')
        break
    
    inputs = tokenizer.apply_chat_template(
                                        [{"role": "user", "content": "假设你是皇帝身边的女人--甄嬛。"},{"role": "user", "content": prompt}],
                                        add_generation_prompt=True,
                                        tokenize=True,
                                        return_tensors="pt",
                                        return_dict=True,
                                        enable_thinking=False
                                    )


    gen_kwargs = {"max_length": 2500, "do_sample": True, "top_k": 1}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
        print(f"\U0001F60DAI嬛儿的回复：\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")